<a href="https://colab.research.google.com/github/MMujtabaX/agentic-ai-journey/blob/main/06_mcp_server/mcp_server_gpt4o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔌 Real MCP with OpenAI GPT-4o-mini
## Build Your Own MCP Server → Expose It → Connect GPT-4o to It

We will:
1. **Build a MCP server** in Python using the `FastMCP` SDK
2. **Run it** inside Colab as a live HTTP service
3. **Expose it** to the internet via ngrok so OpenAI can reach it
4. **Connect GPT-4o** to it using OpenAI's **Responses API** with `type: "mcp"`

GPT-4o will call your server over the actual MCP protocol — the same way it would call GitHub's or Slack's MCP server.

---

## 🎯 Use Case — Developer Research Assistant

You enter a technology (e.g. `LangGraph`, `FastAPI`, `Next.js`). The assistant:

| Step | Tool Called | What Happens |
|------|------------|---------------|
| 1 | `web_search` | Searches the web for info about the technology |
| 2 | `get_github_repo` | Fetches the official GitHub repo's metadata and stats |
| 3 | `read_github_file` | Reads the README from the repo |
| 4 | GPT-4o summarizes | Writes a structured research brief |

**No Anthropic key needed. No paid APIs. Just your OpenAI key + a free ngrok account.**

---

## 🏗️ Real Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                         YOUR COLAB NOTEBOOK                         │
│                                                                     │
│   ┌─────────────────────────────────────────────────────────────┐   │
│   │              FastMCP Server  (Python)                       │   │
│   │              running on localhost:8000/mcp                  │   │
│   │                                                             │   │
│   │   tool: web_search(query)      → DuckDuckGo (free)         │   │
│   │   tool: get_github_repo(...)   → GitHub REST API (free)    │   │
│   │   tool: read_github_file(...)  → GitHub raw content (free) │   │
│   └──────────────────────┬──────────────────────────────────────┘   │
│                          │ uvicorn HTTP                              │
│                          ▼                                           │
│   ┌─────────────────────────────────────────────────────────────┐   │
│   │   pyngrok tunnel  →  https://xxxx.ngrok-free.app/mcp       │   │
│   └──────────────────────┬──────────────────────────────────────┘   │
│                          │ Public HTTPS URL                          │
└──────────────────────────┼──────────────────────────────────────────┘
                           │
                           ▼
          ┌────────────────────────────────────┐
          │   OpenAI Responses API             │
          │   model: gpt-4o                    │
          │   tools: [{ type: 'mcp',           │
          │             server_url: ngrok_url }]│
          └────────────────────────────────────┘
                           │
          GPT-4o calls your MCP server over real MCP protocol
                           │
                           ▼
               📄 Structured Research Brief
```

---

## 🔑 What You Need

| Requirement | Free? | Where |
|-------------|-------|-------|
| OpenAI API key | Paid | [platform.openai.com](https://platform.openai.com) |
| ngrok authtoken | ✅ Free | [dashboard.ngrok.com](https://dashboard.ngrok.com) — signup takes 30 seconds |
| Everything else | ✅ Free | DuckDuckGo search + GitHub public API — no keys needed |

> ⚠️ **ngrok free authtoken**: Go to [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken), sign up for free, and copy your authtoken. Paste it in Cell 3.

---
## 📦 Cell 1 — Install Dependencies

| Package | Role |
|---------|------|
| `mcp[cli]` | FastMCP — the Python MCP server SDK |
| `uvicorn` | ASGI server to run FastMCP over HTTP |
| `pyngrok` | Expose local server to internet via ngrok tunnel |
| `duckduckgo-search` | Free web search — no API key required |
| `httpx` | Async HTTP client for GitHub API calls |
| `openai` | OpenAI SDK — Responses API with MCP support |
| `gradio` | UI |

In [1]:
!pip install "mcp[cli]" uvicorn pyngrok duckduckgo-search httpx openai gradio -q
print("✅ All packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 40.3 MB/s eta 0:00:00
✅ All packages installed.


---
## 📥 Cell 2 — Imports

In [2]:
import os
import time
import threading
import httpx
import json

from mcp.server.fastmcp import FastMCP          # Our MCP server
from duckduckgo_search import DDGS              # Free web search
from pyngrok import ngrok, conf as ngrok_conf   # Public tunnel
from openai import OpenAI                       # Responses API
import gradio as gr                             # UI

import warnings
warnings.filterwarnings("ignore")

print("✅ All imports successful.")

✅ All imports successful.


---
## 🔑 Cell 3 — API Keys

### Getting your free ngrok authtoken (30 seconds):
1. Go to [dashboard.ngrok.com](https://dashboard.ngrok.com)
2. Sign up for free (Google login works)
3. Click **"Your Authtoken"** in the left sidebar
4. Copy and paste it below

The ngrok token lets you create a public HTTPS tunnel from Colab so OpenAI's servers can reach your local MCP server.

In [3]:
# ─── API keys from Colab Secrets (🔑 icon in the left sidebar) ──────────────
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
NGROK_AUTHTOKEN             = userdata.get('NGROK_AUTHTOKEN')   # free at dashboard.ngrok.com

assert os.environ.get("OPENAI_API_KEY"), "❌ Add OPENAI_API_KEY to Colab Secrets"
assert NGROK_AUTHTOKEN, "❌ Add NGROK_AUTHTOKEN to Colab Secrets"

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("✅ OpenAI client ready.")
print("✅ ngrok authtoken set.")

✅ OpenAI client ready.
✅ ngrok authtoken set.


---
## 🔧 Cell 4 — Build the MCP Server

This is the **core of the notebook** — we build a real MCP server using `FastMCP`.

### How FastMCP works

```python
mcp = FastMCP("server-name")   # Create server

@mcp.tool()                    # Register a tool
def my_tool(param: str) -> str:
    return "result"

mcp.run(transport="streamable-http")   # Serve over HTTP
```

FastMCP auto-generates the MCP tool manifest from your Python type hints and docstrings. When GPT-4o connects, it receives a JSON schema for each tool and can call them over HTTP.

### Our 3 tools

| Tool | What it does | External API |
|------|-------------|-------------|
| `web_search` | Search the web for any query | DuckDuckGo (free, no key) |
| `get_github_repo` | Fetch repo metadata, stars, description, topics | GitHub REST API (free, no auth for public repos) |
| `read_github_file` | Read raw content of any file in a public repo | GitHub raw.githubusercontent.com |

> 🔒 The MCP server runs at `localhost:8000/mcp`. ngrok exposes it as a public HTTPS URL. OpenAI calls that URL over the real MCP protocol.

In [4]:
# ─── Create the FastMCP server ────────────────────────────────────────────────
# host="0.0.0.0" → accept connections from ngrok
# stateless_http=True → no session state needed (simpler, OpenAI compatible)
mcp_server = FastMCP(
    name="developer-research-assistant",
    host="0.0.0.0",
    port=8000,
    stateless_http=True,
)


# ════════════════════════════════════════════════════════════
# TOOL 1 — Web Search (DuckDuckGo, completely free)
# ════════════════════════════════════════════════════════════
@mcp_server.tool()
def web_search(query: str, max_results: int = 5) -> str:
    """
    Search the web using DuckDuckGo. Use this to find information,
    documentation, articles, and news about any technology or topic.
    Returns a list of search results with title, URL, and snippet.

    Args:
        query: The search query string
        max_results: Number of results to return (default 5)
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))

        if not results:
            return json.dumps({"error": "No results found", "query": query})

        formatted = []
        for r in results:
            formatted.append({
                "title":   r.get("title", ""),
                "url":     r.get("href",  ""),
                "snippet": r.get("body",  "")[:300]
            })

        return json.dumps({"query": query, "results": formatted})

    except Exception as e:
        return json.dumps({"error": str(e), "query": query})


# ════════════════════════════════════════════════════════════
# TOOL 2 — GitHub Repo Info (GitHub REST API, no auth needed for public repos)
# ════════════════════════════════════════════════════════════
@mcp_server.tool()
def get_github_repo(owner: str, repo: str) -> str:
    """
    Fetch metadata and statistics for a public GitHub repository.
    Returns stars, forks, description, topics, language, and latest release.

    Args:
        owner: GitHub username or organization (e.g. 'langchain-ai')
        repo: Repository name (e.g. 'langgraph')
    """
    try:
        url = f"https://api.github.com/repos/{owner}/{repo}"
        headers = {"Accept": "application/vnd.github+json",
                   "X-GitHub-Api-Version": "2022-11-28"}

        with httpx.Client(timeout=10) as client:
            resp = client.get(url, headers=headers)
            resp.raise_for_status()
            data = resp.json()

        return json.dumps({
            "full_name":    data.get("full_name"),
            "description":  data.get("description"),
            "stars":        data.get("stargazers_count"),
            "forks":        data.get("forks_count"),
            "language":     data.get("language"),
            "topics":       data.get("topics", []),
            "open_issues":  data.get("open_issues_count"),
            "created_at":   data.get("created_at"),
            "updated_at":   data.get("updated_at"),
            "license":      data.get("license", {}).get("name") if data.get("license") else None,
            "homepage":     data.get("homepage"),
            "html_url":     data.get("html_url"),
            "default_branch": data.get("default_branch", "main")
        })

    except httpx.HTTPStatusError as e:
        return json.dumps({"error": f"GitHub API error: {e.response.status_code}",
                           "repo": f"{owner}/{repo}"})
    except Exception as e:
        return json.dumps({"error": str(e)})


# ════════════════════════════════════════════════════════════
# TOOL 3 — Read File from GitHub (raw content, no auth needed)
# ════════════════════════════════════════════════════════════
@mcp_server.tool()
def read_github_file(owner: str, repo: str, path: str, branch: str = "main") -> str:
    """
    Read the raw content of any file in a public GitHub repository.
    Use this to read README files, documentation, configuration files, or source code.
    The most useful path is 'README.md' to understand what a project does.

    Args:
        owner: GitHub username or organization (e.g. 'langchain-ai')
        repo: Repository name (e.g. 'langgraph')
        path: File path in the repo (e.g. 'README.md' or 'docs/index.md')
        branch: Branch name (default 'main', try 'master' if main fails)
    """
    try:
        url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path}"

        with httpx.Client(timeout=15) as client:
            resp = client.get(url)

            # Try 'master' branch if 'main' returns 404
            if resp.status_code == 404 and branch == "main":
                url = f"https://raw.githubusercontent.com/{owner}/{repo}/master/{path}"
                resp = client.get(url)

            resp.raise_for_status()

        # Return first 4000 chars to stay within token limits
        content = resp.text[:4000]
        truncated = len(resp.text) > 4000

        return json.dumps({
            "file":      path,
            "repo":      f"{owner}/{repo}",
            "branch":    branch,
            "content":   content,
            "truncated": truncated,
            "full_length": len(resp.text)
        })

    except httpx.HTTPStatusError as e:
        return json.dumps({"error": f"File not found: {path} ({e.response.status_code})",
                           "suggestion": "Try path='README.md' or branch='master'"})
    except Exception as e:
        return json.dumps({"error": str(e)})


print("✅ MCP Server created: developer-research-assistant")
print("✅ Tools registered:")
print("   🔍 web_search         → DuckDuckGo (free)")
print("   📦 get_github_repo    → GitHub REST API (free)")
print("   📄 read_github_file   → GitHub raw content (free)")

✅ MCP Server created: developer-research-assistant
✅ Tools registered:
   🔍 web_search         → DuckDuckGo (free)
   📦 get_github_repo    → GitHub REST API (free)
   📄 read_github_file   → GitHub raw content (free)


---
## 🚀 Cell 5 — Launch MCP Server + ngrok Tunnel

This cell:
1. Starts the FastMCP server in a **background thread** (so it doesn't block the notebook)
2. Waits for it to be ready
3. Creates an **ngrok HTTPS tunnel** to expose `localhost:8000` publicly
4. Prints the public URL that OpenAI will use

```
Colab localhost:8000/mcp  ──ngrok──▶  https://xxxx.ngrok-free.app/mcp
                                               ▲
                             OpenAI calls this URL over MCP protocol
```

> ⚠️ **Run this cell once per session.** If you restart the runtime, re-run it to get a fresh ngrok URL.

In [5]:
# ── Step 1: Start MCP server in background thread ──────────────────
def _run_mcp():
    """Run FastMCP server – blocks forever (runs in background thread)."""
    mcp_server.run(transport="streamable-http")

server_thread = threading.Thread(target=_run_mcp, daemon=True)
server_thread.start()

print("⏳ Starting MCP server...")
time.sleep(3)  # Give uvicorn time to start

# Verify server is running
try:
    with httpx.Client(timeout=5) as client:
        r = client.get("http://localhost:8000/mcp")
        print(f"✅ MCP server is running (HTTP {r.status_code})")
except Exception as e:
    print(f"⚠️ Server health check: {e} – server may still be starting, continuing...")

# ── Step 2: Create ngrok tunnel ─────────────────────────────────────
print("\n⏳ Creating ngrok tunnel...")

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Kill any existing tunnels first
ngrok.kill()
time.sleep(1)

# Open new tunnel
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

# The MCP endpoint path is /mcp (FastMCP default)
MCP_SERVER_URL = f"{PUBLIC_URL}/mcp"

print(f"\n{'='*60}")
print(f"✅ MCP Server is LIVE and publicly accessible!")
print(f"")
print(f"   Local URL  : http://localhost:8000/mcp")
print(f"   Public URL : {MCP_SERVER_URL}")
print(f"")
print(f"   OpenAI will call: {MCP_SERVER_URL}")
print(f"   GPT-4o will discover tools automatically via MCP protocol")
print(f"{'='*60}")

INFO:     Started server process [1721]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


⏳ Starting MCP server...
INFO:     127.0.0.1:49218 - "GET /mcp HTTP/1.1" 406 Not Acceptable
✅ MCP server is running (HTTP 406)

⏳ Creating ngrok tunnel...

✅ MCP Server is LIVE and publicly accessible!

   Local URL  : http://localhost:8000/mcp
   Public URL : https://quartered-radiation-grout.ngrok-free.dev/mcp

   OpenAI will call: https://quartered-radiation-grout.ngrok-free.dev/mcp
   GPT-4o will discover tools automatically via MCP protocol


---
## 🔍 Cell 6 — Verify the MCP Server (Tool Discovery)

Before connecting GPT-4o, let's verify that the MCP server is correctly exposing its tools.

When an MCP client connects, it first sends a tool discovery request. The server responds with a JSON manifest listing all available tools and their schemas. This is what GPT-4o will receive when it connects.

In [6]:
# ─── Test tool 1: web_search directly (before GPT-4o uses it) ────────────────
print("🧪 Testing MCP tools directly...\n")

# Test web_search
print("Test 1: web_search('LangGraph Python framework')")
result = web_search("LangGraph Python framework", max_results=2)
data = json.loads(result)
for r in data.get("results", []):
    print(f"  • {r['title']}")
    print(f"    {r['url']}")
print()

# Test get_github_repo
print("Test 2: get_github_repo('langchain-ai', 'langgraph')")
result = get_github_repo("langchain-ai", "langgraph")
data = json.loads(result)
print(f"  Name     : {data.get('full_name')}")
print(f"  Stars    : {data.get('stars'):,}")
print(f"  Language : {data.get('language')}")
print(f"  Topics   : {', '.join(data.get('topics', [])[:4])}")
print()

# Test read_github_file
print("Test 3: read_github_file('langchain-ai', 'langgraph', 'README.md')")
result = read_github_file("langchain-ai", "langgraph", "README.md")
data = json.loads(result)
if "content" in data:
    print(f"  File size  : {data['full_length']} chars")
    print(f"  Truncated  : {data['truncated']}")
    print(f"  Preview    : {data['content'][:150].strip()}...")
else:
    print(f"  Error: {data.get('error')}")

print("\n✅ All tools working. Ready to connect GPT-4o.")

🧪 Testing MCP tools directly...

Test 1: web_search('LangGraph Python framework')


/tmp/ipykernel_1721/1424489074.py:27: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Test 2: get_github_repo('langchain-ai', 'langgraph')


  Name     : langchain-ai/langgraph
  Stars    : 29,404
  Language : Python
  Topics   : agents, ai, ai-agents, chatgpt

Test 3: read_github_file('langchain-ai', 'langgraph', 'README.md')
  File size  : 6276 chars
  Truncated  : True
  Preview    : <div align="center">
  <a href="https://www.langchain.com/langgraph">
    <picture>
      <source media="(prefers-color-scheme: dark)" srcset=".github...

✅ All tools working. Ready to connect GPT-4o.


---
## 🧠 Cell 7 — System Prompt + OpenAI Responses API

### OpenAI Responses API vs Chat Completions API

This is **not** `client.chat.completions.create()`. The Responses API is OpenAI's newer interface that natively supports MCP.

| | Chat Completions API | Responses API |
|-|---------------------|---------------|
| **Endpoint** | `client.chat.completions.create()` | `client.responses.create()` |
| **MCP support** | ❌ No | ✅ Yes (`type: "mcp"`) |
| **Tools** | Manual function calling | MCP servers auto-discover tools |
| **Tool execution** | You run the function, send back result | OpenAI calls MCP server directly |
| **Multi-turn** | Manual message history | Built-in with `previous_response_id` |

### The key difference

With **Chat Completions + function calling** (previous notebook):
```
GPT-4o → tells you "call search_jobs" → you run it → you send result back
                                              ▲
                                    You are the middleman
```

With **Responses API + MCP** (this notebook):
```
GPT-4o → directly calls your MCP server over HTTP
                    ▲
         OpenAI is the middleman — you just wait
```

In [7]:
SYSTEM_PROMPT = """
You are a Developer Research Assistant. You have access to 3 MCP tools:
1. web_search       — search the web for any information
2. get_github_repo  — fetch metadata and stats for a GitHub repo
3. read_github_file — read any file from a GitHub repo (most useful: README.md)

When the user gives you a technology or tool to research, follow these steps:

STEP 1 — WEB SEARCH
Call web_search with: "[technology] official documentation overview"
Then call web_search again with: "[technology] use cases examples 2024 2025"

STEP 2 — GITHUB RESEARCH
Identify the official GitHub repo (owner and name) from your search results.
Call get_github_repo to get stats (stars, language, topics).
Call read_github_file with path="README.md" to read the official documentation.
If README.md fails, try path="README.rst" or path="docs/index.md".

STEP 3 — WRITE THE RESEARCH BRIEF
After all tool calls, write a structured brief in this format:

---
## 📦 [Technology Name] — Research Brief

### 🔎 What it is
[2-3 sentence plain-English summary of what the technology does]

### ⭐ GitHub Stats
- Repo: [owner/repo]
- Stars: [number] | Language: [lang] | License: [license]
- Topics: [list]

### 🎯 What it's used for
[3-5 bullet points — specific use cases]

### ✅ Key strengths
[3-4 bullet points]

### ⚠️ Limitations / When NOT to use it
[2-3 bullet points]

### 🚀 Getting started
[1-2 line install command + minimal code snippet from the README]

### 🔗 Key links
[Official docs, GitHub, any important links found]
---

Be specific and use real data from the tools. Do not make things up.
"""


def run_research(technology: str) -> tuple:
    """
    Run the full research workflow using OpenAI Responses API + MCP.

    GPT-4o connects to our MCP server at MCP_SERVER_URL and calls
    the tools autonomously. We just wait for the response.

    Returns: (research_brief_text, tool_calls_log)
    """

    if not technology.strip():
        return "⚠️ Please enter a technology to research.", ""

    print(f"\n{'='*60}")
    print(f"🔬 Researching: {technology}")
    print(f"   MCP Server: {MCP_SERVER_URL}")
    print(f"   Model     : gpt-4o (Responses API)")
    print(f"{'='*60}\n")

    try:
        print("🔌 Connecting OpenAI Responses API to MCP server...")
        print("   GPT-4o will call your MCP tools directly over HTTP.\n")

        # ── THIS is the real MCP call ──────────────────────────────────────────
        # OpenAI connects to MCP_SERVER_URL, discovers tools, calls them itself
        # No manual tool dispatch needed — OpenAI handles everything
        response = openai_client.responses.create(
            model="gpt-4o-mini",
            instructions=SYSTEM_PROMPT,
            input=f"Research this technology for me: {technology}",
            tools=[
                {
                    "type":            "mcp",
                    "server_label":    "developer-research-assistant",
                    "server_url":      MCP_SERVER_URL,   # ← Your ngrok URL
                    "require_approval": "never"          # Auto-approve all tool calls
                }
            ],
        )

        # ── Parse output ───────────────────────────────────────────────────────
        # Responses API returns response.output (list of content items)
        # and response.output_text (all text concatenated)
        final_text = ""
        tool_calls = []

        for item in response.output:
            item_type = getattr(item, "type", "")

            if item_type == "message":
                # Final text response from GPT-4o
                for content in getattr(item, "content", []):
                    if getattr(content, "type", "") == "output_text":
                        final_text += content.text

            elif item_type == "mcp_call":
                # A tool was called by GPT-4o via MCP
                tool_name   = getattr(item, "name", "unknown")
                # arguments can be a dict OR an object — normalise to dict
                raw_args    = getattr(item, "arguments", {})
                tool_input  = raw_args if isinstance(raw_args, dict) else vars(raw_args) if hasattr(raw_args, "_dict_") else {"raw": str(raw_args)}
                tool_output = getattr(item, "output",    "")
                tool_error  = getattr(item, "error",     None)

                status = "✅" if not tool_error else "❌"
                print(f"   {status} MCP tool called: {tool_name}")

                tool_calls.append({
                    "tool":   tool_name,
                    "input":  tool_input,
                    "output": str(tool_output)[:300] if tool_output else str(tool_error)
                })

            elif item_type == "mcp_list_tools":
                # GPT-4o discovered our tool manifest.
                # tools is a list of McpListToolsTool objects — NOT plain dicts.
                # Must use getattr(), NOT .get()
                raw_tools  = getattr(item, "tools", [])
                tool_names = [getattr(t, "name", str(t)) for t in raw_tools]
                print(f"   🔍 GPT-4o discovered MCP tools: {tool_names}")

        # ── Build tool log ─────────────────────────────────────────────────────
        log_lines = [f"### 🔌 MCP Tool Calls — {len(tool_calls)} calls made\n"]
        for i, tc in enumerate(tool_calls, 1):
            log_lines.append(f"*{i}. {tc['tool']}*")
            if isinstance(tc["input"], dict):
                for k, v in tc["input"].items():
                    log_lines.append(f"- {k}: {str(v)[:100]}")
            log_lines.append(f"\n*Result preview:* {tc['output'][:200]}\n")

        tool_log = "\n".join(log_lines)

        if not final_text:
            final_text = "⚠️ No response text generated. Check tool log for details."

        print(f"\n✅ Research complete! {len(tool_calls)} MCP tools called.")
        return final_text, tool_log

    except Exception as e:
        err = f"❌ {type(e).__name__}: {str(e)}"
        print(err)
        return err, ""



print("✅ Responses API function ready.")
print(f"   Will connect GPT-4o to: {MCP_SERVER_URL}")

✅ Responses API function ready.
   Will connect GPT-4o to: https://quartered-radiation-grout.ngrok-free.dev/mcp


---
## 🎨 Cell 9 — Gradio UI

A clean two-tab interface:

| Tab | What you see |
|-----|--------------|
| **📄 Research Brief** | The full structured research output from GPT-4o |
| **🔌 MCP Tool Log** | Every MCP tool GPT-4o called — tool name, inputs, result preview |

In [8]:
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="slate"),
    title="🔬 Developer Research Assistant — Real MCP"
) as demo:

    gr.Markdown(
        f"""
        # 🔬 Developer Research Assistant
        ### Real MCP — OpenAI GPT-4o + Your Own FastMCP Server

        Enter any technology, framework, or library. GPT-4o connects to your MCP server
        at `{MCP_SERVER_URL}` and calls the tools itself.

        > ⏱️ Takes **20–60 seconds** — GPT-4o is making real HTTP calls to your MCP server.
        """
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            gr.Markdown("### 🔍 Research a Technology")

            tech_input = gr.Textbox(
                label="Technology / Framework / Library",
                placeholder="e.g. LangGraph, FastAPI, Next.js, Rust, Svelte...",
                lines=1,
                info="Any public technology with a GitHub repo works."
            )

            run_btn   = gr.Button("🚀 Research", variant="primary", size="lg")
            clear_btn = gr.Button("🗑️ Clear",    variant="secondary", size="sm")

            gr.Markdown(
                f"""
                ---
                **🔌 MCP Server Status**
                - 🟢 Running at `localhost:8000/mcp`
                - 🌐 Public URL: `{MCP_SERVER_URL}`

                **🛠️ Tools Available**
                - `web_search` — DuckDuckGo
                - `get_github_repo` — GitHub API
                - `read_github_file` — GitHub raw
                """
            )

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("📄 Research Brief"):
                    brief_out = gr.Markdown(
                        value=(
                            "*Your research brief will appear here.*\n\n"
                            "GPT-4o will:\n"
                            "1. Search the web for documentation\n"
                            "2. Fetch the GitHub repo stats\n"
                            "3. Read the README\n"
                            "4. Write a structured brief"
                        )
                    )
                with gr.TabItem("🔌 MCP Tool Log"):
                    log_out = gr.Markdown(
                        value=(
                            "*MCP tool calls will appear here.*\n\n"
                            "Shows exactly which tools GPT-4o called via MCP,"
                            " what arguments it passed, and what came back."
                        )
                    )

    gr.Markdown("### 💡 Try These")
    gr.Examples(
        examples=[
            ["LangGraph"],
            ["FastAPI"],
            ["CrewAI"],
            ["Gradio"],
            ["Pydantic"],
            ["Svelte"],
            ["Rust programming language"],
            ["Ollama"],
        ],
        inputs=[tech_input],
        label=""
    )

    run_btn.click(
        fn=run_research,
        inputs=[tech_input],
        outputs=[brief_out, log_out]
    )
    clear_btn.click(
        fn=lambda: ("", "*Results will appear here...*", "*Tool log will appear here...*"),
        inputs=[],
        outputs=[tech_input, brief_out, log_out]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1721/2668010301.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/usr/local/lib/python3.12/dist-packages/gradio/helpers.py:445: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  self.load_input_event = self.dataset.click(
/usr/local/lib/python3.12/dist-packages/gradio/helpers.py:85: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  examples_obj.create()


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


* Running on public URL: https://c47503f3f196e7c63c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1341: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



🔬 Researching: FastAPI
   MCP Server: https://quartered-radiation-grout.ngrok-free.dev/mcp
   Model     : gpt-4o (Responses API)

🔌 Connecting OpenAI Responses API to MCP server...
   GPT-4o will call your MCP tools directly over HTTP.



  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(
  warnings.warn(


   🔍 GPT-4o discovered MCP tools: ['web_search', 'get_github_repo', 'read_github_file']
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: web_search
   ✅ MCP tool called: get_github_repo
   ✅ MCP tool called: get_github_repo
   ✅ MCP tool called: read_github_file

✅ Research complete! 14 MCP tools called.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c47503f3f196e7c63c.gradio.live


  self.proc = None
  self.proc = None


---
## 📚 Cell 10 — What Just Happened? (Deep Dive)

When you clicked **Research**, here is the exact sequence of events:

```
1.  openai_client.responses.create(tools=[{type: 'mcp', server_url: ngrok_url}])
          │
          ▼
2.  OpenAI's servers connect to YOUR ngrok URL
    GET https://xxxx.ngrok-free.app/mcp  ← tool discovery request
          │
          ▼
3.  Your FastMCP server responds with tool manifest:
    {
      tools: [
        { name: 'web_search',      description: '...', parameters: {...} },
        { name: 'get_github_repo', description: '...', parameters: {...} },
        { name: 'read_github_file',description: '...', parameters: {...} }
      ]
    }
          │
          ▼
4.  GPT-4o reads your system prompt + user message + tool schemas
    Decides: 'I should call web_search first'
          │
          ▼
5.  OpenAI calls: POST https://xxxx.ngrok-free.app/mcp
    Body: { tool: 'web_search', arguments: { query: 'LangGraph documentation' } }
          │
          ▼
6.  Your FastMCP server executes web_search() in Python
    DuckDuckGo returns real search results
    Server responds with JSON results
          │
          ▼
7.  GPT-4o reads results, decides: 'Now call get_github_repo'
    OpenAI calls your MCP server again...
          │
          ▼
8.  ... repeat for all 3 tools ...
          │
          ▼
9.  GPT-4o writes the final research brief
    response.output contains the full result
```

### What makes this REAL MCP (not fake)

| | This Notebook | Previous Notebook |
|-|---------------|------------------|
| Protocol | ✅ MCP over HTTP/SSE | ❌ Raw function calling |
| Tool discovery | ✅ GPT-4o fetches manifest from server | ❌ Hardcoded JSON schemas |
| Tool execution | ✅ OpenAI calls your server directly | ❌ You run function, return result |
| Transport | ✅ `streamable-http` (MCP spec) | ❌ Chat completion messages |
| Server | ✅ Real HTTP server (FastMCP + uvicorn) | ❌ No server — just Python functions |

---
## 💡 Cell 11 — Troubleshooting & Extensions

### 🐛 Common Issues

| Error | Cause | Fix |
|-------|-------|-----|
| `AttributeError: 'McpListToolsTool' object has no attribute 'get'` | OpenAI SDK returns objects not dicts in MCP response | Fixed in this notebook — use `getattr(t, 'name')` not `t.get('name')` |
| `ngrok tunnel not found` | Session expired | Re-run Cell 5 to get fresh tunnel |
| `ngrok auth failed` | Wrong authtoken | Get fresh token from dashboard.ngrok.com |
| `MCP server unreachable` | Server crashed | Re-run Cell 4 + 5 |
| `No tools discovered` | Wrong server_url path | Make sure URL ends with `/mcp` |
| `DuckDuckGo rate limit` | Too many searches | Wait 30 seconds and retry |
| `GitHub 403` | Rate limited (60 req/hr unauth) | Add `GITHUB_TOKEN` to headers in Cell 4 |
| Empty research brief | GPT-4o didn't summarize | Check the tool log — tools likely called OK |

### 🔧 Add More Tools to the MCP Server

Adding a tool is just adding a decorated function — no restart needed if added before Cell 5:

```python
@mcp_server.tool()
def get_npm_package(package_name: str) -> str:
    """
    Get info about an NPM package: downloads, version, description.

    Args:
        package_name: The NPM package name (e.g. 'react', 'axios')
    """
    url = f"https://registry.npmjs.org/{package_name}/latest"
    with httpx.Client() as client:
        data = client.get(url).json()
    return json.dumps({
        "name":        data.get("name"),
        "version":     data.get("version"),
        "description": data.get("description"),
        "license":     data.get("license")
    })
```

### 🚀 Next Level Ideas

**Deploy your MCP server permanently** (instead of ngrok):
```bash
# Deploy to Hugging Face Spaces or Railway.app
# Your server gets a permanent URL — no ngrok needed
# Any LLM (GPT-4o, Claude, Gemini) can connect to it forever
```

**Connect Claude to the same server** (if you get an Anthropic key later):
```python
from anthropic import Anthropic
client = Anthropic()
response = client.beta.messages.create(
    model="claude-sonnet-4-20250514",
    mcp_servers=[{ 'type': 'url', 'url': MCP_SERVER_URL, 'name': 'research' }],
    betas=["mcp-client-2025-04-04"]
)
# SAME server, different LLM — that's the power of MCP
```

**Build a CrewAI workflow that uses your MCP server:**
```python
# CrewAI agents can use MCP tools as their tool set
# Agent 1: Researcher (uses web_search + get_github_repo)
# Agent 2: Writer (uses read_github_file + web_search)
# Agent 3: Reviewer (uses web_search for fact-checking)
```